# Tally held-out test: v8 LoRA
v8 adapter (private HF repo renhuang001/tally-v8-lora, root) on the same 126 chunks, prompt, greedy decoding, merge and scorer as the v7 held-out run. bf16 inference. Inference only.

In [ ]:
# === 1. SETUP: install packages, log in, and set paths ===
# Run this after any session restart. Does not load the model yet;
# AlignTune / the TRL fallback will load it for training.

import os
from pathlib import Path

# Kaggle sometimes gives two T4s. Pin to one so the HF Trainer does not wrap the
# 4-bit model in DataParallel. Must run before torch is imported.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Less fragmentation on long, variable-length examples.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

os.environ["HF_HOME"] = "/kaggle/tmp/hf"
os.environ["HF_HUB_CACHE"] = "/kaggle/tmp/hf"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/tmp/hf"
os.makedirs("/kaggle/tmp/hf", exist_ok=True)
os.makedirs("/kaggle/working/tally", exist_ok=True)

# Do not reinstall torch. transformers>=5 matches the extraction notebook.
%pip install -q bitsandbytes peft "transformers>=5"
# v1 failed: PEFT rejected the preinstalled torchao 0.10. The v7 training run had no usable torchao either.
%pip uninstall -y -q torchao

from huggingface_hub import login


def get_hf_token():
    # 1) the notebook's HF_TOKEN Kaggle secret; 2) a private attached dataset
    # file hf_token.txt (renhuang01/tally-hf-token). The token is never printed.
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
        if tok and tok.strip():
            return tok.strip(), "Kaggle secret"
    except Exception as exc:
        print("HF_TOKEN secret not available:", type(exc).__name__)
    for f in sorted(Path("/kaggle/input").rglob("hf_token.txt")):
        tok = f.read_text(encoding="utf-8").strip()
        if tok:
            return tok, "private dataset file"
    raise RuntimeError("No Hugging Face token: attach the HF_TOKEN secret or the private tally-hf-token dataset.")


_hf_tok, _hf_src = get_hf_token()
login(_hf_tok)
os.environ["HF_TOKEN"] = _hf_tok
print("Logged in to Hugging Face via", _hf_src)
del _hf_tok

import torch

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}  "
              f"{torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: no GPU visible. Turn on Settings → Accelerator → GPU T4 x1 and restart.")

print("cuda:", torch.cuda.is_available(), "torch", torch.__version__)
import importlib.metadata as _md
for _p in ("transformers", "trl", "peft", "datasets", "bitsandbytes", "accelerate", "aligntune", "curatorkit"):
    try:
        print(f"{_p}=={_md.version(_p)}")
    except Exception:
        print(_p, "not installed")


In [ ]:

import json, sys, time, traceback, gc
from pathlib import Path
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM

MID = "CohereLabs/tiny-aya-global"
HF_CACHE = "/kaggle/tmp/hf"
RESULTS_DIR = Path("/kaggle/working/heldout"); RESULTS_DIR.mkdir(parents=True, exist_ok=True)
hits = sorted(Path("/kaggle/input").rglob("heldout_messages.jsonl"))
assert hits, "held-out data not attached"
DATA_DIR = hits[0].parent
from huggingface_hub import snapshot_download
_v8 = snapshot_download("renhuang001/tally-v8-lora", allow_patterns=["adapter_config.json","adapter_model.safetensors","tokenizer*","chat_template.jinja","train_metrics.json"], local_dir="/kaggle/working/tally-v8-lora")
print("v8 adapter files:", sorted(p.name for p in Path(_v8).iterdir()))
_ad = [Path(_v8)/"adapter_config.json"]
_unused = []
if not _ad:
    import zipfile
    for z in Path("/kaggle/input").rglob("tally-lora*.zip"):
        zipfile.ZipFile(z).extractall("/kaggle/working/tally-lora-unz")
    _ad = sorted(Path("/kaggle/working/tally-lora-unz").rglob("adapter_config.json"))
ADAPTER_DIR = _ad[0].parent if _ad else DATA_DIR / "tally-lora"
sys.path.insert(0, str(DATA_DIR))
def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
score_rows = load_jsonl(DATA_DIR / "heldout_messages.jsonl")
score_meta = load_jsonl(DATA_DIR / "heldout_meta.jsonl")
answer_key = json.loads((DATA_DIR / "answer_key_heldout.json").read_text(encoding="utf-8"))
from tally_merge import FIELDS, merge_notes
from score_extractions import score_documents, format_summary_table
print("DATA_DIR", DATA_DIR, "| chunks", len(score_rows), "| docs", sorted(answer_key), "| adapter", ADAPTER_DIR.exists())

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# === 6. HELPERS for generation and scoring ===

import re
from collections import defaultdict

from peft import PeftModel


def parse_json(text):
    # Parse the FIRST complete JSON object. (v4: a reply that repeats the
    # object, e.g. '{...}{...}', used to fail with "Extra data".) Same parser
    # for base and fine-tuned, so the comparison stays fair.
    start = text.find("{")
    if start == -1:
        raise json.JSONDecodeError("no JSON object found", text, 0)
    try:
        obj, _ = json.JSONDecoder().raw_decode(text[start:])
        return obj
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.S)
        return json.loads(match.group(0))


def generate_reply(model, tok, prompt, max_new_tokens=900):
    messages = [{"role": "user", "content": prompt}]
    inputs = tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            # stop at <|END_RESPONSE|> or <|END_OF_TURN_TOKEN|>
            eos_token_id=[i for i in (tok.convert_tokens_to_ids("<|END_RESPONSE|>"), tok.eos_token_id) if i is not None],
            pad_token_id=tok.pad_token_id,
        )
    reply_ids = output[0][inputs["input_ids"].shape[1]:]
    return tok.decode(reply_ids, skip_special_tokens=True)


def extract_chunk_text(user_content):
    # Strip the shared extraction prompt, leaving the page-marked chunk.
    marker = "\nPages "
    pos = user_content.find(marker)
    if pos == -1:
        return user_content
    rest = user_content[pos + 1:]  # starts with 'Pages ...'
    # Skip the 'Pages N–M:' header line.
    nl = rest.find("\n")
    return rest[nl + 1:].lstrip() if nl != -1 else rest


def run_val_extraction(model, tok, label):
    # Run every reserved Miro + Bandai chunk, merge per document, and score against the key.
    if not score_rows or not score_meta:
        raise RuntimeError("reserved Miro+Bandai score set is empty or missing")
    per_doc_notes = defaultdict(list)
    per_doc_text = defaultdict(list)
    raw_rows = []
    for i, (row, meta) in enumerate(zip(score_rows, score_meta)):
        doc = meta["doc"]
        prompt = row["messages"][0]["content"]
        print(f"[{label}] {i + 1}/{len(score_rows)}  {doc} chunk {meta.get('chunk_index')}")
        try:
            reply = generate_reply(model, tok, prompt)
            parsed = parse_json(reply)
            err = None
        except Exception as exc:
            reply = locals().get("reply", "")
            parsed = {"_error": f"{type(exc).__name__}: {exc}", "raw": reply[:2000]}
            err = str(exc)
            print("  parse/generate error:", err)
        per_doc_notes[doc].append(parsed)
        per_doc_text[doc].append(extract_chunk_text(prompt))
        raw_rows.append({
            "doc": doc,
            "chunk_index": meta.get("chunk_index"),
            "reply": reply,
            "parsed": parsed,
            "error": err,
        })
        if torch.cuda.is_available() and (i + 1) % 4 == 0:
            torch.cuda.empty_cache()

    merged_by_doc = {}
    for doc, notes in per_doc_notes.items():
        doc_text = "\n\n".join(per_doc_text[doc])
        merged_by_doc[doc] = merge_notes(notes, doc_text=doc_text)

    scores = score_documents(merged_by_doc, answer_key, docs=sorted(merged_by_doc))
    print()
    print(format_summary_table(scores, f"{label} — HELD-OUT (5172, 5432, 4329)"))
    out = {
        "label": label,
        "merged": merged_by_doc,
        "scores": scores,
        "raw": raw_rows,
    }
    path = RESULTS_DIR / f"{label}_heldout_scores.json"
    path.write_text(json.dumps(out, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    print("wrote", path)
    return out


def load_base_model():
    tok = AutoTokenizer.from_pretrained(MID, cache_dir=HF_CACHE)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    # Inference in bfloat16, matching the extraction notebook, unless memory is tight.
    try:
        model = AutoModelForCausalLM.from_pretrained(
            MID,
            dtype=torch.bbfloat16,
            device_map={"": 0},
            cache_dir=HF_CACHE,
        )
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(
            MID,
            torch_dtype=torch.bbfloat16,
            device_map={"": 0},
            cache_dir=HF_CACHE,
        )
    model.eval()
    return tok, model


def release(*objs):
    for obj in objs:
        try:
            del obj
        except Exception:
            pass
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# v2: base already scored in the v7 held-out run; skip it to save GPU time.
base_results = None
print("base skipped (scored in version 1)")


In [ ]:

t0 = time.time()
ft_results = None
try:
    ft_tok, base_for_adapter = load_base_model()
    import importlib.util; print("torchao present:", importlib.util.find_spec("torchao") is not None)
    ft_model = PeftModel.from_pretrained(base_for_adapter, str(ADAPTER_DIR))
    ft_model.eval()
    if hasattr(ft_model, "config"):
        ft_model.config.use_cache = True
    ft_results = run_val_extraction(ft_model, ft_tok, "v8")
except Exception as exc:
    print("v8 scoring failed:", type(exc).__name__, exc); traceback.print_exc()
finally:
    ft_model = None; base_for_adapter = None; free_gpu()
print("v8 done in %.1f min" % ((time.time() - t0) / 60))


In [ ]:

def overall_line(name, s):
    o = s["overall"]
    return (f"{name:<10} stated={o['stated_accuracy']:.1%} ({o['stated_correct']}/{o['stated_n']})  acc+abs={o['accuracy_incl_abstention']:.1%}  "
            f"correct={o['correct']} wrong={o['wrong']} conflict={o['conflict-unresolved']} abstained={o['correctly-abstained']} hallucinated={o['hallucinated']}")
bs = base_results["scores"] if base_results else None
fs = ft_results["scores"] if ft_results else None
if bs: print(overall_line("base", bs)); print(format_summary_table(bs, "BASE — held-out"))
if fs: print(overall_line("v8", fs)); print(format_summary_table(fs, "V8 — held-out"))
if bs and fs:
    for a, b in zip(bs["pairs"], fs["pairs"]):
        if a["category"] != b["category"]:
            print(f"  {a['doc']} / {a['field']}: {a['category']} -> {b['category']} | gold={a['gold_value']!r} base={a['extract_value']!r} v8={b['extract_value']!r}")
comparison = {"heldout_docs": sorted(answer_key), "n_chunks": len(score_rows),
              "base": bs, "v8": fs,
              "base_merged": base_results["merged"] if base_results else None,
              "v8_merged": ft_results["merged"] if ft_results else None}
for p in (RESULTS_DIR / "heldout_comparison.json", Path("/kaggle/working/heldout_comparison.json")):
    p.write_text(json.dumps(comparison, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("wrote heldout_comparison.json")
